In [1]:
import pprint
from langchain.tools import tool
from langchain.chat_models import init_chat_model
from langchain_groq import ChatGroq
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from typing_extensions import TypedDict
from langchain.agents.structured_output import ToolStrategy, ProviderStrategy
import os

from dotenv import load_dotenv

load_dotenv("C:\\Users\\socgen\\ML\\agentic_ai_and_ops\\langchain_day5\\.env")

True

In [2]:
model_gr_lamma = init_chat_model("llama-3.3-70b-versatile",
                        api_key=os.environ["GROQ_API_KEY"],
                        model_provider="groq",
                        # base_url="https://api.groq.com/openai/v1",
                        max_tokens=1000, temperature=0.0)

model_or_paid_gpt_luna_pro = init_chat_model("openai/gpt-5.6-luna-pro",
                        api_key=os.environ["OPENROUTER_API_KEY"],
                        model_provider="openrouter",
                        base_url="https://openrouter.ai/api/v1",
                        max_tokens=100, temperature=0.0)

model_or_free_nvidia = init_chat_model("nvidia/nemotron-3-ultra-550b-a55b:free",
                        api_key=os.environ["OPENROUTER_API_KEY"],
                        model_provider="openrouter",
                        base_url="https://openrouter.ai/api/v1",
                        max_tokens=1000, temperature=0.0)


model_or_free = init_chat_model("openrouter/free",
                        api_key=os.environ["OPENROUTER_API_KEY"],
                        model_provider="openrouter",
                        base_url="https://openrouter.ai/api/v1",
                        max_tokens=1000, temperature=0.0)


model_ollama = init_chat_model("ollama:gemma4:latest",
                                max_tokens=200, 
                                temperature=0.0)

In [3]:
product_review_json_schema = {
    "$schema": "http://json-schema.org/draft-07/schema#",
    "title": "ProductReview",
    "type": "object",
    "properties": {
        "product_name": {"type": "string"},
        "review_text": {"type": "string"},
        "rating": {"type": "integer", "enum": [1, 2, 3, 4, 5]},
        "shipping_sentiment": {
            "type": "string",
            "enum": ["positive", "negative", "neutral"],
        },
        "pricing_sentiment": {
            "type": "string",
            "enum": ["positive", "negative", "neutral"],
        },
        "quality_sentiment": {
            "type": "string",
            "enum": ["positive", "negative", "neutral"],
        },
    },
    "required": [
        "product_name",
        "review_text",
        "rating",
        "shipping_sentiment",
        "pricing_sentiment",
        "quality_sentiment",
    ],
    "additionalProperties": False,
}


In [9]:
agent = create_agent(
    model=model_gr_lamma,
    tools=[],
    response_format=product_review_json_schema,
    # response_format=ToolStrategy(
    #     product_review_json_schema
    # )
    )

In [10]:
result=agent.invoke({
    "messages": [{"role": "user", "content": "Analyze this review: 'Great TShirt: 5 out of 5 stars. slow shipping, and expensive'"}]
})

In [11]:
result

{'messages': [HumanMessage(content="Analyze this review: 'Great TShirt: 5 out of 5 stars. slow shipping, and expensive'", additional_kwargs={}, response_metadata={}, id='4c53c607-045e-44dc-bae6-0db6cb2229a4'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'n5vn0tkvg', 'function': {'arguments': '{"pricing_sentiment":"negative","product_name":"TShirt","quality_sentiment":"positive","rating":5,"review_text":"Great TShirt: 5 out of 5 stars. slow shipping, and expensive","shipping_sentiment":"negative"}', 'name': 'ProductReview'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 69, 'prompt_tokens': 351, 'total_tokens': 420, 'completion_time': 0.122662904, 'completion_tokens_details': None, 'prompt_time': 0.01797467, 'prompt_tokens_details': None, 'queue_time': 0.053218999, 'total_time': 0.140637574}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_dae98b5ecb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'lo

In [7]:
result

{'messages': [HumanMessage(content="Analyze this review: 'Great TShirt: 5 out of 5 stars. slow shipping, and expensive'", additional_kwargs={}, response_metadata={}, id='74ab96c2-10e2-4bab-bb14-943f6f3f1fee'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '8ya9jgepv', 'function': {'arguments': '{"pricing_sentiment":"negative","product_name":"TShirt","quality_sentiment":"positive","rating":5,"review_text":"Great TShirt: 5 out of 5 stars. slow shipping, and expensive","shipping_sentiment":"negative"}', 'name': 'ProductReview'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 69, 'prompt_tokens': 351, 'total_tokens': 420, 'completion_time': 0.12816599, 'completion_tokens_details': None, 'prompt_time': 0.090702437, 'prompt_tokens_details': None, 'queue_time': 0.058209118, 'total_time': 0.218868427}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_dae98b5ecb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'lo

In [7]:
result["structured_response"]

{'product_name': 'TShirt',
 'review_text': 'Great TShirt: 5 out of 5 stars. slow shipping, and expensive',
 'rating': 5,
 'shipping_sentiment': 'negative',
 'pricing_sentiment': 'negative',
 'quality_sentiment': 'positive'}